# 01 — Inventory & sanity check

First step of the pipeline. Confirms what's on disk, what CRS / GSD / extent every product carries, and flags any silent inconsistencies between products (notably the **Pix4D-local vs WGS84 vertical reference** issue).

Logic lives in `src/drone_reserve/inventory.py`; this notebook only orchestrates.

Inputs read: everything under `data/` (metadata only — no full raster reads).

Outputs written: `outputs/01_inventory/inventory.json` + a markdown summary.

In [1]:
# Requires `pip install -e .` once from the repo root. See README setup.
from pathlib import Path
from drone_reserve.inventory import scan_directory, print_summary

# Locate the repo root so we don't hard-code an absolute path.
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "pyproject.toml").is_file():
        REPO = parent
        break
else:
    raise RuntimeError("Couldn't locate repo root (no pyproject.toml found upward).")

In [2]:
DATA = REPO / "data"
OUT  = REPO / "outputs" / "01_inventory"
OUT.mkdir(parents=True, exist_ok=True)

scan = scan_directory(DATA)
print_summary(scan, DATA)


=== Rasters (14) ===
  drone\pastizal_120m\products\Pastizal_120m_20250521_dsm.tif
    size=184.4 MB  11330x13379px  bands=1 dtype=float32 nodata=-10000.0  GSD=0.034 m  17.99 ha
    crs=EPSG:32721  overviews=none
  drone\pastizal_120m\products\Pastizal_120m_20250521_dtm.tif
    size=594.5 KB  566x668px  bands=1 dtype=float32 nodata=-10000.0  GSD=0.689 m  17.95 ha
    crs=EPSG:32721  overviews=none
  drone\pastizal_120m\products\Pastizal_120m_20250521_transparent_mosaic_group1.tif
    size=170.3 MB  11330x13379px  bands=4 dtype=uint8 nodata=None  GSD=0.034 m  17.99 ha
    crs=EPSG:32721  overviews=none
  drone\talar_120m\products\Talar_120m_20250521_dsm.tif
    size=170.2 MB  11417x11293px  bands=1 dtype=float32 nodata=-10000.0  GSD=0.034 m  15.13 ha
    crs=EPSG:32721  overviews=none
  drone\talar_120m\products\Talar_120m_20250521_dtm.tif
    size=625.2 KB  570x564px  bands=1 dtype=float32 nodata=-10000.0  GSD=0.685 m  15.09 ha
    crs=EPSG:32721  overviews=none
  drone\talar_120m\pro

## Cross-product consistency checks

Things the bare inventory doesn't catch — but matter before any analysis runs.

In [3]:
import rasterio
import numpy as np

def z_window_stats(path, win_px=1024):
    """Read a centred WIN_PXxWIN_PX window and return (min, median, max) of valid pixels."""
    with rasterio.open(path) as src:
        w = min(src.width, win_px); h = min(src.height, win_px)
        col_off = (src.width - w) // 2; row_off = (src.height - h) // 2
        win = rasterio.windows.Window(col_off, row_off, w, h)
        a = src.read(1, window=win, masked=True).compressed()
    if not a.size:
        return (np.nan, np.nan, np.nan)
    return (float(a.min()), float(np.median(a)), float(a.max()))

products = [
    ("Pix4D talar DSM",  DATA / "drone/talar_120m/products/Talar_120m_20250521_dsm.tif"),
    ("Root  talar DSM",  DATA / "talar_dsm.tif"),
    ("Pix4D talar DTM",  DATA / "drone/talar_120m/products/Talar_120m_20250521_dtm.tif"),
    ("Root  talar DTM",  DATA / "talar_dtm.tif"),
    ("Pix4D past  DSM",  DATA / "drone/pastizal_120m/products/Pastizal_120m_20250521_dsm.tif"),
    ("Root  past  DSM",  DATA / "pastizal_dsm.tif"),
    ("GNSS  DTM",        DATA / "gnss_dtm.tif"),
]
print(f"{'product':18s} {'min':>8s} {'median':>8s} {'max':>8s}")
for label, p in products:
    lo, mid, hi = z_window_stats(p)
    print(f"{label:18s} {lo:8.2f} {mid:8.2f} {hi:8.2f}")

product                 min   median      max


Pix4D talar DSM     -104.14  -101.33   -95.86


Root  talar DSM       22.46    25.27    30.74
Pix4D talar DTM     -111.87  -103.84  -100.07
Root  talar DTM       14.73    22.76    26.53


Pix4D past  DSM      -99.03   -95.34   -87.60


Root  past  DSM       19.14    22.83    30.57
GNSS  DTM             22.22    22.60    24.87


**Interpretation.** The Pix4D-original rasters live in Pix4D's local vertical frame (Z ≈ −100 m). The root-level copies have been vertically registered to real elevation (Z ≈ +22 m) by applying the shift documented in `data/talar.csv` / `data/pastizal.csv`. The LAS point clouds are still in the Pix4D local frame and will need the same shift before they can be compared with dGNSS or with the vertically-registered rasters.

Calibration shifts (from the CSVs):

- Talar:    ΔX = +1.11 m,  ΔY = +1.80 m,  ΔZ = +126.60 m
- Pastizal: ΔX = −2.37 m,  ΔY = +0.11 m,  ΔZ = +118.17 m

In [4]:
# Save a machine-readable inventory snapshot for downstream notebooks.
import json
from dataclasses import asdict

payload = {
    "rasters":   [asdict(r) for r in scan.rasters],
    "las_files": [asdict(l) for l in scan.las_files],
    "vectors":   [asdict(v) for v in scan.vectors],
    "image_counts": scan.image_counts,
}
out_json = OUT / "inventory.json"
out_json.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
print(f"wrote {out_json}")

wrote C:\Users\paco_\OneDrive\Escritorio\repos\drone_reserve\outputs\01_inventory\inventory.json


## Gaps relative to the planned pipeline

See the gap report appended to this notebook output / printed to the README. Update the list here as inputs arrive.